# Momentum

The ideas are in the slides. Here we build momentum three ways and ask each version the same two questions:

- Does it beat the market? A CAPM regression.
- Is it anything more than UMD? A regression on the market and UMD.

**The plan**

1. Stock momentum, from our panel (1981–2000)
2. 🎯 Industry momentum, 1927–2026: your turn. Then back to the slides
3. 🎯 Challenge (homework): volatility-managed momentum

📑 **Slides for this class:** [Momentum (PDF)](https://amoreira2.github.io/UG54/slides/momentum.pdf)

<iframe src="../../slides/momentum.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

panel = pd.read_parquet(f"{BASE}/panel_backbone_1980_2000.parquet")   # CRSP stocks, monthly
ind   = pd.read_parquet(f"{BASE}/ind49_monthly_1926.parquet")         # 49 industries, monthly
ind_d = pd.read_parquet(f"{BASE}/ind49_daily_1926.parquet")           # the same 49 industries, daily
fac   = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")       # Mkt-RF, RF, UMD, monthly

sharpe   = lambda x: x.mean() / x.std() * np.sqrt(12)
drawdown = lambda x: ((1 + x).cumprod() / (1 + x).cumprod().cummax() - 1).min()

def regress(y, factors):
    """Regress a monthly return series on columns of `fac`. Prints the alpha per year, its t-stat and the betas."""
    r = sm.OLS(y, sm.add_constant(fac.loc[y.index, factors])).fit()
    betas = '   '.join(f"{f} {r.params[f]:+.2f}" for f in factors)
    print(f"on {' + '.join(factors):12s}  alpha {r.params['const']*12:+6.1%}/yr  (t {r.tvalues['const']:+.2f})   {betas}   R2 {r.rsquared:.2f}")
    return r

print(f"stocks      {panel.date.min():%Y-%m} to {panel.date.max():%Y-%m}")
print(f"industries  {ind.index[0]:%Y-%m} to {ind.index[-1]:%Y-%m}, daily too")
print(f"factors     {fac.index[0]:%Y-%m} to {fac.index[-1]:%Y-%m}")

---

## 1. Stock momentum

The rule from the slides: rank stocks on their return from $t-12$ to $t-2$, buy the top decile, short the
bottom decile. Value-weighted, NYSE breakpoints.

Four steps. Read each one before you run it.

In [ ]:
p = panel[panel.shrcd.isin([10, 11]) & panel.exchcd.isin([1, 2, 3])].sort_values(['permno', 'date']).copy()
p['1+ret'] = 1 + p['ret']

# 1. the signal: each stock's return from t-12 to t-2
p['cum11'] = (p.groupby('permno')['1+ret'].rolling(11).apply(np.prod, raw=True)
                .reset_index(level=0, drop=True) - 1)
p['signal'] = p.groupby('permno')['cum11'].shift(2)      # ends at t-2: skip the most recent month
p['me_lag'] = p.groupby('permno')['me'].shift(1)         # weights you know at the start of the month

# 2. each month, cut into deciles on NYSE breakpoints
d = p.dropna(subset=['signal', 'ret', 'me_lag']).copy()
def decile(x):
    edges = np.unique(np.quantile(x.loc[x.exchcd == 1, 'signal'], np.linspace(0, 1, 11)))
    edges[0], edges[-1] = -np.inf, np.inf
    return pd.cut(x['signal'], edges, labels=False, duplicates='drop')
d['decile'] = d.groupby('date', group_keys=False).apply(decile)

# 3. the value-weighted return of each decile
dec = d.groupby(['date', 'decile']).apply(lambda x: np.average(x['ret'], weights=x['me_lag'])).unstack()

# 4. winners minus losers
wml = dec[9] - dec[0]
print(f"winners minus losers, {wml.index[0]:%Y-%m} to {wml.index[-1]:%Y-%m}:  "
      f"mean {wml.mean()*12:+.1%}/yr   Sharpe {sharpe(wml):.2f}")

In [ ]:
ax = (dec.mean() * 12).plot.bar(color='grey')
ax.set_xticklabels([f"{k + 1}" for k in dec.columns], rotation=0)
ax.set_xlabel('decile of past return (1 = losers, 10 = winners)'); ax.set_ylabel('average return per year')
plt.show()

regress(wml, ['Mkt-RF'])
regress(wml, ['Mkt-RF', 'UMD']);

- Which end does more of the work, the winners or the losers?
- The market beta is close to zero. Why?
- Ours loads about 1.5 on UMD, not 1. Why more? (UMD buys the top 30% and shorts the bottom 30%.)

---

## 2. 🎯 Industry momentum, 1927–2026

Is momentum about stocks, or about industries? Lets build the industry version and run the same two regressions.

`ind` has monthly returns on 49 industries from July 1926. A few industries start later, so their early
months are NaN.

**The rule**

- signal: each industry's return from $t-12$ to $t-2$
- each month, buy the 5 industries with the highest signal and short the 5 with the lowest, equal-weighted
- only rank industries that have a signal that month

Use AI. Call the result `IM`: one return per month, dated by the month you earn it.

In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below ----

IM = ____

In [ ]:
#@title 🔒 Check
signal = ((1 + ind).rolling(11).apply(np.prod, raw=True) - 1).shift(2)     # t-12 to t-2
top    = signal.rank(axis=1, ascending=False) <= 5
bottom = signal.rank(axis=1, ascending=True) <= 5
IM_ref = (ind.where(top).mean(axis=1) - ind.where(bottom).mean(axis=1)).dropna()

print(f"reference  {IM_ref.index[0]:%Y-%m} to {IM_ref.index[-1]:%Y-%m}   mean {IM_ref.mean()*12:+.1%}/yr   Sharpe {sharpe(IM_ref):.2f}")
try:
    j = pd.concat([IM, IM_ref], axis=1).dropna()
    print(f"yours      {IM.index[0]:%Y-%m} to {IM.index[-1]:%Y-%m}   mean {IM.mean()*12:+.1%}/yr   Sharpe {sharpe(IM):.2f}"
          f"   correlation with the reference {j.corr().iloc[0, 1]:.3f}")
except Exception as e:
    print("yours      not built yet:", type(e).__name__)
IM = IM_ref          # from here on everyone uses the reference

In [ ]:
both = pd.concat([IM.rename('industry momentum'), fac['UMD']], axis=1).dropna()
ax = np.log(1 + both).cumsum().plot(color=['black', 'tab:blue'])
ax.set_ylabel('cumulative log return'); plt.show()

print(f"correlation with UMD {both.corr().iloc[0, 1]:.2f}")
regress(IM, ['Mkt-RF'])
regress(IM, ['Mkt-RF', 'UMD']);

- When does industry momentum crash? Is it the same months as UMD?
- How much of the CAPM alpha is left once you control for UMD?
- So, is industry momentum a different trade from stock momentum?

---

## ⏸ Back to the slides

The challenge below is homework. It uses the second half of the slides.

---

## 🎯 Challenge: volatility-managed momentum

*Homework — due before the next class.*

Momentum's crashes come when volatility is high, and volatility is predictable. So hold less of it after a
volatile month:

$$IM^{\sigma}_t \;=\; \frac{c}{RV_{t-1}}\; IM_t$$

- $RV_{t-1}$ is the realized variance of industry momentum's **daily** returns in month $t-1$:
  $\sum_d (r_d - \bar r)^2$ over the days of that month
- $c$ is the constant that gives $IM^{\sigma}$ the same volatility as `IM` over the same months. It sets the
  scale; it does not change the Sharpe ratio

`ind_d` has the daily returns of the 49 industries. On every day of month $t$ you hold the portfolio you
formed at the end of month $t-1$: the same 5 winners and 5 losers as `IM` that month, equal-weighted.

*Hint: `top` and `bottom` from the check cell tell you which industries you hold each month.*

### Q1 — Build it

Report its Sharpe ratio and its maximum drawdown. Compare with `IM`.

> **📌 Required variable names:**
> ```python
> imv_sharpe = ____     # Sharpe ratio of the volatility-managed version
> imv_maxdd  = ____     # its maximum drawdown, negative decimal (-0.30 means -30%)
> ```

In [ ]:
# Your work here


imv_sharpe = ____
imv_maxdd  = ____

print(imv_sharpe, imv_maxdd)

### Q2 — The two regressions

Run the CAPM and the market + UMD regressions on your volatility-managed series. Report both alphas per year,
as decimals (0.05 means +5%/yr).

> **📌 Required variable names:**
> ```python
> imv_alpha_capm = ____   # alpha on the market, per year
> imv_alpha_umd  = ____   # alpha on the market and UMD, per year
> ```

In [ ]:
# Your work here


imv_alpha_capm = ____
imv_alpha_umd  = ____

print(imv_alpha_capm, imv_alpha_umd)

### Q3 — The memo

> **📝 Your task — maximum eight sentences.**
>
> Your PM runs industry momentum. Explain what the volatility-managed version would have done in its worst
> years, and why it works: what is predictable here, and what is not? Say what the market + UMD alpha tells
> you that the CAPM alpha does not. Finish with one reason you might not earn this in practice.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["imv_sharpe", "imv_maxdd", "imv_alpha_capm", "imv_alpha_umd", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L10_Momentum_v2_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")